# Step 5: Self-Similar Finite-Rank Residuals and Physical Norms

Bound the residual and finite-rank norms of the supplied field-even inverse trials. The retained spectral fields $\psi_i$ define their forcing; the loaded inverse fields are separate inputs.

Run from `notebooks/` with `dtype = Interval{Float64}` selected before loading the module. The padded residual grids and pairwise convection arrays require substantial memory.

**Inputs:** the Step 1 profile `data/generated/UP_divfree.mat`; the paired Step 4B files `data/generated/rhs.mat` and `rhs_bound.mat`; `data/external/phi_even/UP_phi_$(ind-1).mat`; `data/external/eig.mat`; and the normalized Step 2 eigenfunction `data/generated/up_eig_corrected.mat` for mixed forcing.

**Output:** `data/generated/step5_even_matrices.mat`. Step 6 performs the odd calculation independently.

This notebook supplies the paper's $\Theta_r^U$, $\Theta_\phi^U$, $\Theta_{\nabla\phi}^U$, $\Theta_Q^U$, mixed-forcing bound $\Theta_F^U$, and the two aggregates entering $M_3^U$. Their numerical inputs appear under `eq:45-input-rU`, `eq:45-input-phi-U`, `eq:45-input-gradphi-U`, `eq:45-input-GU`, `eq:45-input-FU`, `eq:45-input-product-U`, and `eq:45-input-grad-phi-U`.

Read in order: inputs → global derivative priors → inner and outer residual Grams → gradient bounds → inverse-field Grams → mixed forcing → pairwise convection → save. See the [symmetry note](../docs/technical_notes/symmetry_and_domains.md#even-finite-rank-residual), [operator note](../docs/technical_notes/ns_terms.md), and [quadrature note](../docs/technical_notes/matrix_func.md).


In [1]:
# Load the project module with its already selected numeric type.
import Pkg
Pkg.activate("..")
include("../src/NS_Numerics.jl")
using .NS_Numerics


  Activating project at `~/Changhe/3d-navier-stokes-nonuniqueness-main-run`


Using dtype = IntervalArithmetic.Interval{Float64}
Number of threads = 56


## Load matched inputs and identify the retained fields

| Input | Fields used here | Role |
|---|---|---|
| `UP_divfree.mat` | Prepared `u1,u2,u3,p`, scale and conversion data | Background `UP`, boundary set 2; `divfree=true` skips reconstruction |
| `rhs.mat` | `rhs_even` | Weighted forcing samples; `K=size(rhs_tensor,3)`, following the supplied retained modes |
| `phi_even/UP_phi_$(ind-1).mat` | Velocity/pressure coefficients | External approximate inverse field, boundary set 3, indexed `0:K-1` on disk |
| `eig.mat` | `eig_val_even` | Eigenvalues for Gram scaling |
| `up_eig_corrected.mat` | Normalized velocity coefficients | Unit physical L² eigenfunction for mixed forcing; no norm recomputation |
| `rhs_bound.mat` | `rhs_prior_even`, `rhs_bound_even` | Order-ten β priors and refined order-eight directional bounds, loaded in the next block |

The $\psi_i$ are the retained combinations of Bessel/spherical-harmonic basis fields reconstructed in Step 4B. Its exported forcing is $wQ\psi_i$, with Q evaluated in its own radial argument $x_Q=(\pi/2)r$. The $\phi_i$ loaded here denote raw trial fields $\Phi_i$. The manuscript inverse fields are $\overline\phi_i^U=-\Phi_i$, so $G^U=-\Phi\mathcal B^*$ with $\mathcal B_i=Q\psi_i/\nu_i$. Their mass, gradient, and same-sign convection Grams are unchanged by this common sign. This notebook neither reconstructs $\psi_i$ nor solves an eigenproblem. Its `UP_phi` loader uses the default divergence-free reconstruction of the angular velocity coefficients; later operators read those coefficients.

The sample tensor has axes `(β sample, θ sample, retained mode, component)`. The expected even shape is `(12021,6021,K,3)`, including ten saved ghost layers on both grid axes. The derivative files use `(component, mode, order 0:10, singleton)` for `rhs_prior_even` and `(component, order 0:8, direction, mode)` for `rhs_bound_even`. The code repacks these into one `3 × K` matrix per prior order and one `9 × 2` refined table per mode. The [RHS note](../docs/technical_notes/finite_rank_rhs.md#7-save-the-rhs-samples-and-bounds) describes these layouts.

Mode order, scales, component frame, eigenvalue normalization, and sample grids must agree across these files. In particular, the generated RHS samples and bounds form a matched pair; the sample and derivative files are used as a pair. `M0_UP,M1_UP` describe the background, while `M0,M1` describe the even inverse fields. The bounds `freq_mixed`, `freq_phi`, `freq_pair`, and `freq_forcing` apply to their respective terms and include a `+10` allowance. `freq_forcing` uses the actual phi and normalized-v coefficient dimensions.

In [ ]:
# Load the matched even forcing and approximate inverse fields; k=10 seeds order-eight refinement.

# Load the prepared background coefficients without reconstructing their divergence-free relation again.
UP = from_mat_file("../data/generated/UP_divfree.mat", 2; domain="frequency", divfree=true)
M0_UP, M1_UP = size(UP["u3"].freq_func)
up, _ = from_mat_file("../data/generated/up_eig_corrected.mat", 1; domain="frequency", divfree=true)
M0_v, M1_v = size(up["u3"].freq_func)

# Load the right-hand side tensor.
# Each slice rhs_tensor[:,:,ind,:] stores the already weighted forcing for one inverse field.
# The sign convention and physical residual are defined in the following Markdown section.
rhs_tensor = read_matrix("../data/generated/rhs.mat", ["rhs_even"])
K = size(rhs_tensor, 3)
# K = 1

# Define domain parameters.
R = dt(32.0)
L0, L1 = Pi / dt(2), Pi / dt(2)
# Sobolev regularity order
k = 10
padding = derivative_padding(k-2)
# Scaling factor relating r and β coordinates.
a = UP.scl_fac / R
Beta = atan(inv(a))

# Pair each external inverse field φ_i with its saved forcing w Q ψ_i in the same retained-mode order.
UP_phi_lst = [] # stores φ_i (basis functions)
rhs_lst = [] # stores rhs_i

for ind in 1:K
    println("i = $ind / $K")

    # Load the approximate inverse velocity/pressure coefficients, not the Bessel field ψ_i.
    # These are approximate inverse solutions φ_i associated with retained spectral functions ψ_i.
    UP_phi = from_mat_file("../data/external/phi_even/UP_phi_$(ind-1).mat", 3; domain="frequency")
    push!(UP_phi_lst, UP_phi)

    # Construct rhs_i as a VectorFunc in physical space.
    rhs = VectorFunc(UP.scl_fac, "space"; e1=rhs_tensor[:, :, ind, 1], 
        e2=rhs_tensor[:, :, ind, 2], e3=rhs_tensor[:, :, ind, 3])
    push!(rhs_lst, rhs)
end

# Release the tensor after constructing the independent RHS fields.
rhs_tensor = nothing
GC.gc(true)

M0, M1 = size(UP_phi_lst[1]["u3"].freq_func)
freq_phi = dt.([2*M0 + 10, 2*M1 + 10])
freq_pair = dt.([4*M0 + 10, 4*M1 + 10])
freq_mixed = dt.([
    2*(M0_UP + M0) + 10,
    2*(M1_UP + M1) + 10
])

freq_forcing = dt.([2*(M0+M0_v)+10, 2*(M1+M1_v)+10])

# Only eigenvalues are needed for Gram scaling; Step 4B already constructed the forcing.
eig_val = read_matrix("../data/external/eig.mat", ["eig_val_even"]);


i = 1 / 18
i = 2 / 18
i = 3 / 18
i = 4 / 18
i = 5 / 18
i = 6 / 18
i = 7 / 18
i = 8 / 18
i = 9 / 18
i = 10 / 18
i = 11 / 18
i = 12 / 18
i = 13 / 18
i = 14 / 18
i = 15 / 18
i = 16 / 18
i = 17 / 18
i = 18 / 18


18×1 Matrix{Interval{Float64}}:
 [11.7322, 11.7323]_com
  [3.6671, 3.66711]_com
  [2.48721, 2.48722]_com
  [2.02926, 2.02927]_com
  [1.79143, 1.79144]_com
  [1.53447, 1.53448]_com
  [1.31621, 1.31622]_com
  [1.13297, 1.13298]_com
  [0.998851, 0.998852]_com
  [0.908409, 0.90841]_com
  [0.804006, 0.804007]_com
  [0.713874, 0.713875]_com
  [0.654635, 0.654636]_com
  [0.645263, 0.645264]_com
  [0.601469, 0.60147]_com
  [0.545289, 0.54529]_com
  [0.493792, 0.493793]_com
  [0.47449, 0.474491]_com

## Physical fields, the radial weight, and the residual sign

Let $\kappa=\texttt{UP.scl\_fac}$, $\rho=|x|$ be physical radius, and $R=32$ the support radius of Q. The compactification and inner cutoff are

$$
\rho=\kappa\tan\beta,\qquad
r=\frac{\rho}{R}=a\tan\beta,\qquad
a=\frac{\kappa}{R},\qquad
\mathrm{Beta}=\arctan(R/\kappa).
$$

Here r is the normalized radius used to construct the Bessel forcing in Step 4B. Physical space is $0\leq\beta<\pi/2$; $\beta=\pi/2$ represents infinity. The functions stored in `UP` and `UP_phi_lst[i]` are compactified velocity/pressure coefficients:

$$
U=\cos\beta\,\widehat U,\qquad
\phi_i=\cos\beta\,\widehat\phi_i,\qquad
p_i=\cos\beta\,\widehat p_i.
$$

All derivatives below are physical derivatives. Define $D=(I+x\cdot\nabla)/2$ and

$$
B_i=D\phi_i+\Delta\phi_i+\nabla p_i,\qquad
C_i=(U\cdot\nabla)\phi_i+(\phi_i\cdot\nabla)U,\qquad
w(\beta)=\frac{\sin\beta}{\cos^2\beta}.
$$

`get_linear` returns $wB_i$, including a **positive** pressure-gradient block; the notebook subtracts this whole output. The two `mul_A_gradB` calls together return $wC_i$. Consequently the code's operator-side `lfs` and full residual are

$$
\texttt{lfs}_i=w(C_i-B_i),\qquad
\texttt{err}_i=wE_i,\qquad
E_i=Q\psi_i+C_i-B_i.
$$

Equivalently, with $\mathcal A_U(\phi_i,p_i)=B_i-C_i$, this is the forcing-minus-operator residual $E_i=Q\psi_i-\mathcal A_U(\phi_i,p_i)$. This explicit definition fixes the pressure and operator signs without inferring them from a paper symbol. There is **no eigenvalue shift** in this Step 5 residual; `eig_val` enters only the Gram denominators below. See [the implementation conventions](../docs/technical_notes/ns_terms.md#9-get_linear-weighted-physical-linear-terms).

| Sampled object | Represented physical quantity | Purpose |
|---|---|---|
| `rhs_lst[i]` | $wQ\psi_i$ on the inner formula's grid | Forcing and forcing Gram matrix |
| `lfs` in the prior loop | $w(C_i-B_i)$ | Global derivative priors |
| `err_lst[i]` | $wE_i$ | Refined residual and residual Gram matrix |
| `I_UP_phi` | $w\phi_i=\tan\beta\,\widehat\phi_i$ | Physical velocity L² norm |
| `nabla_UP_i_phy` | Nine entries of $\nabla\phi_i$ | Physical gradient L∞ bound |
| `UP_i_nabla_UP_j_lst[i,j]` | $w(\phi_i\cdot\nabla)\phi_j$ | Pairwise physical L² norms |

### Why the weighted arrays give physical inner products

The radial Jacobian is $\rho^2\,d\rho=\kappa^3w(\beta)^2\,d\beta$. For same-sector physical fields F and H, if $f=wF$ and $h=wH$, the notebook's weighted inner product on a radial interval I is

$$
\langle F,H\rangle_I
=4\pi\kappa^3\int_I\int_0^{\pi/2}
 f(\beta,\theta)\cdot h(\beta,\theta)\sin\theta\,d\theta\,d\beta.
$$

`inner_product` supplies $\sin\theta$ and $4\pi\kappa^3$. The factor $4\pi$ combines the azimuthal integral $2\pi$ with reflection of the half meridian; it is not a radial doubling. The radial weight is already in the arrays and must not be inserted again. There is no additional $R^3$ multiplier in these calls. Gradient L∞ bounds instead use `weight_flag=false`, so they concern the unweighted physical gradient.

## Global operator priors: why β extends to π

The unpadded physical grid uses `N0=6000`, `N1=3000` on $[0,\pi/2]^2$. The prior grid `β_prior` instead uses **12000 intervals on $[0,\pi]$**, preserving the same β mesh spacing. Only the `lfs` prior is evaluated on this wider interval; `β_pt`, `L0`, and all physical integration regions retain their original meaning.

To justify Bernstein bounds, a sampled supremum must control the trigonometric polynomial on its full period. For the even inverse field, write the operator-side residual as $F=V-P$: V collects the velocity convection minus pressure-free linear terms, and P is the weighted pressure gradient. V is even about both $\beta=0$ and $\beta=\pi/2$; P is even about zero and odd about $\pi/2$ in its two nonzero components. Therefore F is even about zero, but generally has no common parity about $\pi/2$. Cancellation between V and P on the physical quarter interval need not persist after reflection.

Origin evenness and $2\pi$ periodicity make $[0,\pi]$ sufficient for the full-period absolute supremum, including derivatives. The wider grid is an analytic extension of the assembled trigonometric expression, not additional physical space. See [the even residual derivation](../docs/technical_notes/symmetry_and_domains.md#even-finite-rank-residual).

The other frequency-based calculations retain the physical quarter grid because their assembled components have definite radial parity at both endpoints:

| Quantity | At β=0 | At β=π/2 |
|---|---|---|
| Physical $\nabla\phi_i$ | Even | Odd |
| Weighted identity $w\phi_i$ | Even | Even |
| Weighted $(\phi_i\cdot\nabla)\phi_j$ | Even | Odd |

Absolute values of every derivative admit the same reflection reduction. These statements concern the complete regularized operator expressions; one must not bound the singular-looking weight w separately at $\pi/2$.

Angular reduction holds throughout this notebook. The field-even vector components use angular codes `11,00,01`, with parities `(even,even)`, `(odd,odd)`, and `(odd,even)` at $\theta=0,\pi/2$. Convection between two even fields, the linear operator, and Q multiplication preserve this sector; gradient entries also have definite angular signs. Componentwise absolute derivative suprema are therefore represented on $[0,\pi/2]$. Products of two same-sector components are even about the equator, which separately justifies the doubled angular integrals. See [the angular symmetry rules](../docs/technical_notes/symmetry_and_domains.md#angular-symmetries).

`freq_mixed=[2*(M0_UP+M0)+10,2*(M1_UP+M1)+10]` bounds the background/inverse operator terms. `freq_phi=[2*M0+10,2*M1+10]` is for single inverse fields, and `freq_pair=[4*M0+10,4*M1+10]` is for inverse/inverse convection. The operator prior/refinement uses `freq_mixed`. The prior pass corrects sampled maxima and applies frequency powers without finite differences or padding. Its beta column fills `lfs_β_norms_prior`; adding `rhs_β_norms_prior` gives the inner residual priors `err_β_norms_prior`. Compute all frequency ceilings from the actual loaded background and inverse dimensions.

In [3]:
# Operator/RHS derivative priors needed by both residual regions.
rhs_norms, lfs_β_norms_prior, err_β_norms_prior = let
    # Bound the complete weighted operator-side expression on the origin-even half-period.
    # Construct physical evaluation grid in β and θ coordinates.
    # These grid points are used to apply the linear operator and compute residuals.
    N0, N1 = 6000, 3000
    β_pt = grid(L0, N0)
    θ_pt = grid(L1, N1)
    # Prior only: lfs is even at β=0 but mixes parities at β=π/2.
    β_prior = grid(Pi, 2*N0)

    # Construct matrix representation of the base linear operator.
    # This corresponds to L_base(φ) and does NOT include UP-dependent convection terms.
    lin_mat_phi = get_operator_matrix(UP_phi_lst[1], β_prior, θ_pt, "linear");

    # Compute UP-dependent convection terms.
    # These correspond to:
    #     mul_A_gradB(UP, φ) + mul_A_gradB(φ, UP),
    # and together form the convection component of the operator.
    lfs_lst = vec(mul_A_gradB(UP, UP_phi_lst, β_prior, θ_pt))
    add_equal!.(lfs_lst, vec(mul_A_gradB(UP_phi_lst, UP, β_prior, θ_pt)))

    rhs_β_norms_prior_mat = read_matrix("../data/generated/rhs_bound.mat", ["rhs_prior_even"])
    rhs_β_norms_prior = [rhs_β_norms_prior_mat[:, :, i, 1] for i in 1: k+1];
    lfs_β_norms_prior = [zeros(eltype(M), size(M)) for M in rhs_β_norms_prior]
    rhs_norms = read_matrix("../data/generated/rhs_bound.mat", ["rhs_bound_even"]); 
    rhs_norms = [[rhs_norms[:,i,j,l] for i in 1: k-1, j in 1:2] for l in 1:K]# stores W^{k-2,∞} norm bounds of rhs_i

    for ind in 1:K
        println("i = $ind / $K")

        # Construct full left-hand side operator application
        # Note sign convention follows code implementation
        lfs = lfs_lst[ind] - apply(lin_mat_phi, UP_phi_lst[ind])

        # Compute W^{k,∞} priors for the operator-side combination.
        lfs_norm = norm_lst(lfs, "W$(k)Inf"; paras=freq_mixed, domain=[Pi,L1])
        for i in 1: k+1, j in 1: 3
            lfs_β_norms_prior[i][j, ind] = lfs_norm[i, 1][j]
        end
    end

    # Combined bound used later in residual inner product estimates
    err_β_norms_prior = lfs_β_norms_prior .+ rhs_β_norms_prior;

    # Keep only the small derivative-bound tables for subsequent cells.
    rhs_norms, lfs_β_norms_prior, err_β_norms_prior
end
GC.gc(true)


i = 1 / 18
i = 2 / 18
i = 3 / 18
i = 4 / 18
i = 5 / 18
i = 6 / 18
i = 7 / 18
i = 8 / 18
i = 9 / 18
i = 10 / 18
i = 11 / 18
i = 12 / 18
i = 13 / 18
i = 14 / 18
i = 15 / 18
i = 16 / 18
i = 17 / 18
i = 18 / 18


## Refine the residual on the two physical radial regions

The physical forcing $Q\psi_i$ is supported in $\rho\leq R$, corresponding to $\beta\leq\mathrm{Beta}$. The residual is therefore evaluated separately on the support and its complement:

$$
E_i=
\begin{cases}
Q\psi_i+C_i-B_i,&0\leq\beta\leq\mathrm{Beta},\\
C_i-B_i,&\mathrm{Beta}<\beta\leq\pi/2.
\end{cases}
$$

The grid parameters below are numbers of **interior intervals**, not sample counts. The angular interval is always $[0,\pi/2]$.

| Stage | β interval | Interior intervals $(N_0,N_1)$ | Samples before trimming |
|---|---|---|---|
| Inner residual and RHS | $[0,\mathrm{Beta}]$ | `(12000,6000)` | `12021 × 6021` |
| Outer residual | $[\mathrm{Beta},\pi/2]$ | `(9000,6000)` | `9021 × 6021` |

### Why ten layers and order-ten priors are needed

`k=10` supplies prior derivatives through order ten; the target `W$(k-2)Inf` is order eight, which supplies the derivatives required for quadrature. Backward finite-difference refinement through order eight needs

$$
\texttt{padding}=\texttt{derivative\_padding}(8)=1+2+3+4=10
$$

ghost samples at **each end of each coordinate**. These layers cover the nested stencils, not extra physical integration volume. The seed bounds for orders nine and ten must hold throughout the padded interval. The [padding derivation](../docs/technical_notes/matrix_func.md#why-refinement-proceeds-from-high-order-to-low-order) explains the accumulated stencil widths.

Step 4B already saves ten layers in `rhs.mat`; its order-ten `rhs_prior_even` covers those stencils. Keep these samples while adding the newly evaluated operator terms. Only after `norm_lst(...; padding=padding)` finishes are ten layers removed from both `err_lst[ind]` and `rhs_lst[ind]`. The resulting `12001 × 6001` arrays are the physical inner quadrature grid. See [the producer's padding workflow](../docs/technical_notes/finite_rank_rhs.md#how-the-workflow-padding-is-chosen).

Near `Beta`, the inner ghost samples use the smooth continuation of the inner Bessel/Q formula. They do not differentiate the piecewise zero extension across the support boundary. The outer operator formula is refined independently, then the two integrals are added.

`Linf_to_paras(...; ind=ind)` selects one retained mode and the **special refinement path**. The inner region uses `err_β_norms_prior`; the exterior uses the operator-only `lfs_β_norms_prior`. It first refines β derivatives from explicit priors, then forms angular seeds and transverse corrections from the refined β bounds of orders zero and two using angular Bernstein. Thus no radial Fourier bandwidth is asserted for the mixed Bessel/trigonometric residual on either restricted interval. The zero radial frequency in this special call is not a claim that the residual is constant in β. Fresh parameter packages are made for every call because angular refinement updates them.

Each `err_norms[ind]` is a `9 × 2` table of three-component bound vectors. Entry `[m+1,j][c]` bounds the pure order-m derivative of component c, with `j=1` for β and `j=2` for θ. These supply the Gram quadrature errors; they are not one combined Sobolev norm.

## Assemble the inner Gram matrices

Let $\lambda_i=\texttt{eig\_val[i]}$, $f_i=Q\psi_i/\lambda_i$, and $\Omega=\{x:|x|\leq R\}$. These $\lambda_i$ are the retained generalized eigenvalues associated with $\psi_i$, not the unstable eigenvalue from the eigenpair notebook. The active inner-region calls compute

$$
GQ_{ij}=\langle f_i,f_j\rangle_{L^2(\Omega)},\qquad
GR^{\rm inner}_{ij}=\langle E_i,E_j\rangle_{L^2(\Omega)}.
$$

The forcing is not divided by $\lambda_i$ when the residual is assembled. Only `GQ` divides its pairwise integral by $\lambda_i\lambda_j$. `GR` has no eigenvalue divisor.

`GQ` and `GR` are symmetric: only the upper triangles are integrated, then copied. `GR` at this stage includes the inner contribution only. The later outer loop completes it in place.

Every active `inner_product` call supplies `W_norms_lst`. The vector wrapper bounds the eighth derivatives of the product, including $\sin\theta$, and adds the quadrature-error allowance with the same physical $4\pi\kappa^3$ factor as the sampled integral. The unpadded interval counts are divisible by six, as required by the composite Newton–Cotes rule. See [the weighted quadrature derivation](../docs/technical_notes/matrix_func.md#from-one-dimensional-quadrature-to-a-weighted-inner-product).
Once GQ is complete, the cell prints the candidate C_U bound and directly certifies the inline threshold `dt("5.24693")` using interval LDLᵀ. Only GQ and GR are returned. This is not a measured eigenvalue; no GQ Frobenius diagnostic is also computed.


In [4]:
# Inner residual and the first contributions to GQ and GR.
GQ, GR = let
    # Keep the ten saved RHS layers until the assembled residual has been refined through order eight.
    # Refine grid on the near-origin region β ∈ [0, Beta] for tighter residual evaluation.
    N0 = 12000
    N1 = 6000
    β_pt = grid(Beta, N0, padding)
    θ_pt = grid(L1, N1, padding)

    # Base linear part of the operator on this refined grid (UP-independent part).
    lin_mat_phi = get_operator_matrix(UP_phi_lst[1], β_pt, θ_pt, "linear");
    # Build residuals err_i on β ∈ [0, Beta].
    # We first form the UP-dependent convection/coupling contribution:
    #   C_UP(ϕ_i) = mul_A_gradB(UP, ϕ_i) + mul_A_gradB(ϕ_i, UP)
    # Then we add rhs_i - L_base(ϕ_i) so that err_i matches the full linear system residual
    # under the sign convention used in this notebook.
    err_lst = vec(mul_A_gradB(UP, UP_phi_lst, β_pt, θ_pt))
    add_equal!.(err_lst, vec(mul_A_gradB(UP_phi_lst, UP, β_pt, θ_pt)))

    for ind in 1:K
        println("i = $ind / $K")
    
        rhs = rhs_lst[ind]
        # Keep all ten saved RHS ghost layers through derivative refinement.
        # Trim only after residual derivative refinement below.

        # Base linear part: L_base(ϕ_i)
        lfs = apply(lin_mat_phi, UP_phi_lst[ind])
        # Residual assembly: err_i += rhs_i - L_base(ϕ_i)
        # (err_lst already contains the UP-dependent term)
        add_equal!(err_lst[ind], rhs - lfs)
    end

    err_norms = []
    for ind in 1:K
        paras = Linf_to_paras(err_β_norms_prior, k, dt.([0, freq_mixed[2]]); ind=ind)
        push!(err_norms, norm_lst(err_lst[ind], "W$(k-2)Inf"; paras=paras, domain=[Beta,L1], padding=padding))
        trim_border!(err_lst[ind], padding)
        trim_border!(rhs_lst[ind], padding)
    end

    # err_norms_copy = deepcopy(err_norms)
    # for i=1:9, j=1:3
    #     err_norms[1][i, 1][j] = min(err_norms[1][i, 1][j], err_β_norms_prior[i][j, 1])
    #     # println("i=$i, j=$j, prior=$(err_β_norms_prior[i][j, 1]), post=$(err_norms[1][i, 1][j])")
    # end

    # Integrate weighted inner-region products with derivative-based quadrature allowances.
    # Assemble Gram matrices on the region where rhs is supported (β ∈ [0, Beta]).
    # At this stage, GR contains only the contribution from the region where rhs ≠ 0.
    GQ = zeros(dtype, K, K)
    GR = zeros(dtype, K, K)

    for i in 1:K, j in i:K
        println("i = $i / $K, j = $j / $K")
        # RHS Gram matrix (rhs support region)
        GQ[i, j] = inner_product(rhs_lst[i], rhs_lst[j]; W_norms_lst=[rhs_norms[i], 
            rhs_norms[j]], domain=[Beta, L1]) / (eig_val[j] * eig_val[i])
        # GQ[i, j] = inner_product(rhs_lst[i], rhs_lst[j]; domain=[Beta, L1]) / (eig_val[j] * eig_val[i])
        
        # Residual Gram matrix (first-region contribution)
        GR[i, j] = inner_product(err_lst[i], err_lst[j]; W_norms_lst=[err_norms[i], 
            err_norms[j]], domain=[Beta, L1])
    
        # Gram matrices are symmetric; fill lower triangle by copying.
        if i != j
            GQ[j, i] = GQ[i, j]
            GR[j, i] = GR[i, j]
        end
    end

    println("C_U bound: ", dt("5.24693"))
    @assert verify_symmetric_spectrum(GQ, dt("5.24693"))

    GQ, GR
end

# RHS samples have now served their only remaining purpose.
rhs_lst = nothing
GC.gc(true)


i = 1 / 18
i = 2 / 18
i = 3 / 18
i = 4 / 18
i = 5 / 18
i = 6 / 18
i = 7 / 18
i = 8 / 18
i = 9 / 18
i = 10 / 18
i = 11 / 18
i = 12 / 18
i = 13 / 18
i = 14 / 18
i = 15 / 18
i = 16 / 18
i = 17 / 18
i = 18 / 18
i = 1 / 18, j = 1 / 18
i = 1 / 18, j = 2 / 18
i = 1 / 18, j = 3 / 18
i = 1 / 18, j = 4 / 18
i = 1 / 18, j = 5 / 18
i = 1 / 18, j = 6 / 18
i = 1 / 18, j = 7 / 18
i = 1 / 18, j = 8 / 18
i = 1 / 18, j = 9 / 18
i = 1 / 18, j = 10 / 18
i = 1 / 18, j = 11 / 18
i = 1 / 18, j = 12 / 18
i = 1 / 18, j = 13 / 18
i = 1 / 18, j = 14 / 18
i = 1 / 18, j = 15 / 18
i = 1 / 18, j = 16 / 18
i = 1 / 18, j = 17 / 18
i = 1 / 18, j = 18 / 18
i = 2 / 18, j = 2 / 18
i = 2 / 18, j = 3 / 18
i = 2 / 18, j = 4 / 18
i = 2 / 18, j = 5 / 18
i = 2 / 18, j = 6 / 18
i = 2 / 18, j = 7 / 18
i = 2 / 18, j = 8 / 18
i = 2 / 18, j = 9 / 18
i = 2 / 18, j = 10 / 18
i = 2 / 18, j = 11 / 18
i = 2 / 18, j = 12 / 18
i = 2 / 18, j = 13 / 18
i = 2 / 18, j = 14 / 18
i = 2 / 18, j = 15 / 18
i = 2 / 18, j = 16 / 18
i = 2 / 18, j = 17

## Add the complement of the forcing support

The outer grid covers $\beta\in[\mathrm{Beta},\pi/2]$, or physical radii $\rho\geq R$. Its residual contains only $w(C_i-B_i)$; no RHS samples are added. It is evaluated with ten ghost layers and refined through order eight before trimming to `9001 × 6001` samples.

The code uses the sharper operator-only `lfs_β_norms_prior` here because the physical RHS vanishes identically outside its support. The inner-region combined prior remains unchanged. Origin reflection and periodicity extend the operator prior to the small halo beyond $\pi/2$.

`domain=[L0-Beta,L1]` supplies interval **lengths** for derivative spacings and quadrature; the samples themselves were evaluated with the β offset `+Beta`. After the following addition,

$$
GR_{ij}=GR^{\rm inner}_{ij}+\langle E_i,E_j\rangle_{L^2(\mathbb R^3\setminus\Omega)}
=\langle E_i,E_j\rangle_{L^2(\mathbb R^3)}.
$$

`GQ` needs no outer contribution because $f_j=0$ there. 
After the outer contribution is complete, the cell immediately computes `K7_U=sqrt(sum(GQ .* GR))`, prints the interval, and checks its upper endpoint against `dt("0.000453683")`. Only this trace estimate is used for K7_U.


In [5]:
# Add the exterior contribution to the existing GR in place.
K7_U = let
    # Refine the RHS-free outer residual on its own padded physical subinterval.
    # Switch to the complementary region β ∈ [Beta, L0],
    # where rhs = 0 and only the operator-dependent part contributes to the residual.
    N0 = 9000
    N1 = 6000
    β_pt = grid(L0 - Beta, N0, padding) .+ Beta
    θ_pt = grid(L1, N1, padding)

    # Base linear operator matrix on this region.
    lin_mat_phi = get_operator_matrix(UP_phi_lst[1], β_pt, θ_pt, "linear");

    # Construct residual on the region where rhs = 0.
    # Since rhs vanishes here, the residual reduces to
    #     err_i = (UP-dependent terms) - L_base(φ_i)
    err_lst = vec(mul_A_gradB(UP, UP_phi_lst, β_pt, θ_pt))
    add_equal!.(err_lst, vec(mul_A_gradB(UP_phi_lst, UP, β_pt, θ_pt)))

    for ind in 1:K
        println("i = $ind / $K")

        # Base linear part
        lfs = apply(lin_mat_phi, UP_phi_lst[ind])
        # Residual = convection − base linear part
        minus_equal!(err_lst[ind], lfs)
    end

    err_norms_pt2 = []
    for ind in 1:K
        paras = Linf_to_paras(lfs_β_norms_prior, k, dt.([0, freq_mixed[2]]); ind=ind)
        push!(err_norms_pt2, norm_lst(err_lst[ind], "W$(k-2)Inf"; paras=paras, domain=[L0-Beta, L1], padding=padding))
        trim_border!(err_lst[ind], padding)
    end

    # Complete GR in place; the existing GR contains only the inner contribution.
    # Add the residual Gram matrix contribution from the region where rhs = 0.
    # This completes the full-domain residual Gram matrix GR.
    # GR2 = zeros(dtype, K, K)

    for i in 1:K, j in i:K
        println("i = $i / $K, j = $j / $K")
        GR[i, j] += inner_product(err_lst[i], err_lst[j]; W_norms_lst=[err_norms_pt2[i], 
            err_norms_pt2[j]], domain=[L0-Beta, L1])
        # GR[i, j] += inner_product(err_lst[i], err_lst[j]; domain=[L0-Beta, L1])
    
        # Gram matrices are symmetric; fill lower triangle by copying.
        if i != j
            GR[j, i] = GR[i, j]
        end
    end
    K7_U = sqrt(sum(GQ .* GR))
    println("K7_U: ", K7_U)
    @assert sup(K7_U) <= inf(dt("0.000453683"))
    K7_U
end
GC.gc(true)


i = 1 / 18
i = 2 / 18
i = 3 / 18
i = 4 / 18
i = 5 / 18
i = 6 / 18
i = 7 / 18
i = 8 / 18
i = 9 / 18
i = 10 / 18
i = 11 / 18
i = 12 / 18
i = 13 / 18
i = 14 / 18
i = 15 / 18
i = 16 / 18
i = 17 / 18
i = 18 / 18
i = 1 / 18, j = 1 / 18
i = 1 / 18, j = 2 / 18
i = 1 / 18, j = 3 / 18
i = 1 / 18, j = 4 / 18
i = 1 / 18, j = 5 / 18
i = 1 / 18, j = 6 / 18
i = 1 / 18, j = 7 / 18
i = 1 / 18, j = 8 / 18
i = 1 / 18, j = 9 / 18
i = 1 / 18, j = 10 / 18
i = 1 / 18, j = 11 / 18
i = 1 / 18, j = 12 / 18
i = 1 / 18, j = 13 / 18
i = 1 / 18, j = 14 / 18
i = 1 / 18, j = 15 / 18
i = 1 / 18, j = 16 / 18
i = 1 / 18, j = 17 / 18
i = 1 / 18, j = 18 / 18
i = 2 / 18, j = 2 / 18
i = 2 / 18, j = 3 / 18
i = 2 / 18, j = 4 / 18
i = 2 / 18, j = 5 / 18
i = 2 / 18, j = 6 / 18
i = 2 / 18, j = 7 / 18
i = 2 / 18, j = 8 / 18
i = 2 / 18, j = 9 / 18
i = 2 / 18, j = 10 / 18
i = 2 / 18, j = 11 / 18
i = 2 / 18, j = 12 / 18
i = 2 / 18, j = 13 / 18
i = 2 / 18, j = 14 / 18
i = 2 / 18, j = 15 / 18
i = 2 / 18, j = 16 / 18
i = 2 / 18, j = 17

## Bound the physical gradient

The gradient operator explicitly uses `weight_flag=false`; `nabla_UP_i_phy` contains all nine spherical-frame entries of $\nabla\phi_i$. `norm_lst(...,"LInf")` returns a bound for the absolute supremum of each entry. If $b_{i,ab}$ denotes that entry's upper bound, the code then forms

$$
\left(\sum_i\sum_{a,b=1}^3 b_{i,ab}^2\right)^{1/2}
\ \geq\ 
\left(\sum_i\|\nabla\phi_i\|_{L^\infty(\mathbb R^3;F)}^2\right)^{1/2},
$$

where F denotes the pointwise Frobenius matrix norm. Entrywise suprema may occur at different locations, so the computed expression is a conservative bound, not an exact gradient maximum. It also bounds the aggregate using pointwise operator norms.

`freq_phi=[2*M0+10,2*M1+10]` supplies the single-field frequency ceilings. The gradient calculation uses the unpadded `6001 × 3001` physical grid; no residual or RHS is included. Once `linf_norms` is complete, `phi_grad=norm(linf_norms)` is printed and immediately checked against `dt("0.369356")`. The next cell computes the whole-space inverse-field Gram matrices.


In [6]:
# Physical gradient supremum bounds.
linf_norms = let
    # Independent whole-space grid (not the preceding exterior grid).
    N0, N1 = 6000, 3000
    β_pt = grid(L0, N0)
    θ_pt = grid(L1, N1)
    # Bound physical gradients without radial weighting.
    # Construct gradient operator matrix
    D_phy_mat = get_operator_matrix(UP_phi_lst[1], β_pt, θ_pt, "gradient"; weight_flag=false)
    linf_norms = []

    for ind in 1:K
        UP_i = UP_phi_lst[ind]
        # Compute the physical gradient
        nabla_UP_i_phy = apply(D_phy_mat, UP_i; new_keys=["A11", "A12", "A13", "A21", "A22", "A23", "A31", "A32", "A33"])
        # Compute the L∞ norm of the gradient
        UP_i_W1_norm = norm_lst(nabla_UP_i_phy, "LInf"; paras=freq_phi, domain=[L0,L1])

        push!(linf_norms, norm(UP_i_W1_norm))
    end

    linf_norms
end
phi_grad = norm(linf_norms)
println("phi gradient Linf aggregate: ", phi_grad)
@assert sup(phi_grad) <= inf(dt("0.369356"))
GC.gc(true)


phi gradient Linf aggregate: [0.369355, 0.369356]_com


### Whole-space Gram matrices for the $U_2$ estimates

We use `G_phi` for $(\langle\phi_i,\phi_j\rangle_{L^2(\mathbb R^3)})_{ij}$ and `G_Dphi` for $(\langle\nabla\phi_i,\nabla\phi_j\rangle_{L^2(\mathbb R^3)})_{ij}$. Integration by parts gives $G_{D\phi,ij}=-\langle\phi_i,\Delta\phi_j\rangle$: the finite even modes are $O(\rho)$ at the origin and $O(\rho^{-2})$ at infinity, so the boundary terms vanish.

The `"laplacian"` operator gives the weighted physical Laplacian directly, with drift and pressure omitted. The identity and Laplacian fields carry the same weight required by `inner_product`, including its $4\pi\kappa^3$ normalization. Their even radial parities and definite angular reflection parities allow whole-quarter-domain W8 priors with the existing frequency allowance. The local grid is independent of the subdomain grids used later. 

The `verify_gram_spectrum` calls check the candidates `A0_U=4.28080` and `A1_U=4.79779` through PSD upper matrices and final interval LDL tests. Exact Gram structure and valid interval enclosures are prerequisites; the helper does not first require the original interval Gram to pass a positive-definite check. The triangle inequality implies `B0_U=1+A0_U=5.28080`, but the code does not print or save this derived value. Only the two Grams are returned. See [the Gram certificate](../docs/technical_notes/matrix_func.md#symmetric-and-gram-spectral-certificates).


In [ ]:
# Whole-space identity/Laplacian Gram matrices for the even inverse fields.
G_phi, G_Dphi = let
    β_pt = grid(L0, 6000)
    θ_pt = grid(L1, 3000)

    I_mat = get_operator_matrix(UP_phi_lst[1], β_pt, θ_pt, "identity")
    Lap_mat = get_operator_matrix(UP_phi_lst[1], β_pt, θ_pt, "laplacian")

    # Cache each expensive sampled field exactly once.
    I_UP_phi_lst = []
    Lap_UP_phi_lst = []
    UP_phi_Wk∞_lst = []
    Lap_UP_phi_Wk∞_lst = []

    for i in 1:K
        println("Field $i / $K")
        I_UP_phi = apply(I_mat, UP_phi_lst[i])
        Lap_UP_phi = apply(Lap_mat, UP_phi_lst[i])
        push!(I_UP_phi_lst, I_UP_phi)
        push!(Lap_UP_phi_lst, Lap_UP_phi)
        push!(UP_phi_Wk∞_lst,
            norm_lst(I_UP_phi, "W$(k-2)Inf"; paras=freq_phi, domain=[L0,L1]))
        push!(Lap_UP_phi_Wk∞_lst,
            norm_lst(Lap_UP_phi, "W$(k-2)Inf"; paras=freq_phi, domain=[L0,L1]))
    end

    G_phi = zeros(dtype, K, K)
    G_Dphi = zeros(dtype, K, K)

    for i in 1:K, j in i:K
        println("i = $i / $K, j = $j / $K")
        G_phi[i,j] = inner_product(
            I_UP_phi_lst[i], I_UP_phi_lst[j];
            W_norms_lst=[UP_phi_Wk∞_lst[i], UP_phi_Wk∞_lst[j]],
            domain=[L0,L1])
        G_Dphi[i,j] = -inner_product(
            I_UP_phi_lst[i], Lap_UP_phi_lst[j];
            W_norms_lst=[UP_phi_Wk∞_lst[i], Lap_UP_phi_Wk∞_lst[j]],
            domain=[L0,L1])
        if i != j
            G_phi[j,i] = G_phi[i,j]
            G_Dphi[j,i] = G_Dphi[i,j]
        end
    end

    println("A0_U bound: ", dt("4.28080"))
    @assert verify_gram_spectrum(GQ, G_phi, dt("4.28080")^2)

    println("A1_U bound: ", dt("4.79779"))
    @assert verify_gram_spectrum(GQ, G_Dphi, dt("4.79779")^2)

    G_phi, G_Dphi
end
GC.gc(true)


Field 1 / 18
Field 2 / 18
Field 3 / 18
Field 4 / 18
Field 5 / 18
Field 6 / 18
Field 7 / 18
Field 8 / 18
Field 9 / 18
Field 10 / 18
Field 11 / 18
Field 12 / 18
Field 13 / 18
Field 14 / 18
Field 15 / 18
Field 16 / 18
Field 17 / 18
Field 18 / 18
i = 1 / 18, j = 1 / 18
i = 1 / 18, j = 2 / 18
i = 1 / 18, j = 3 / 18
i = 1 / 18, j = 4 / 18
i = 1 / 18, j = 5 / 18
i = 1 / 18, j = 6 / 18
i = 1 / 18, j = 7 / 18
i = 1 / 18, j = 8 / 18
i = 1 / 18, j = 9 / 18
i = 1 / 18, j = 10 / 18
i = 1 / 18, j = 11 / 18
i = 1 / 18, j = 12 / 18
i = 1 / 18, j = 13 / 18
i = 1 / 18, j = 14 / 18
i = 1 / 18, j = 15 / 18
i = 1 / 18, j = 16 / 18
i = 1 / 18, j = 17 / 18
i = 1 / 18, j = 18 / 18
i = 2 / 18, j = 2 / 18
i = 2 / 18, j = 3 / 18
i = 2 / 18, j = 4 / 18
i = 2 / 18, j = 5 / 18
i = 2 / 18, j = 6 / 18
i = 2 / 18, j = 7 / 18
i = 2 / 18, j = 8 / 18
i = 2 / 18, j = 9 / 18
i = 2 / 18, j = 10 / 18
i = 2 / 18, j = 11 / 18
i = 2 / 18, j = 12 / 18
i = 2 / 18, j = 13 / 18
i = 2 / 18, j = 14 / 18
i = 2 / 18, j = 15 / 18
i = 2 

### Mixed forcing and its whole-space trace bound

For the normalized corrected v and loaded inverse trials $\Phi_j$, form

$$
F_j=(\Phi_j\cdot\nabla)v+(v\cdot\nabla)\Phi_j,
\qquad GF_{ij}=\langle F_i,F_j\rangle.
$$

The paper defines $\Theta_F^U$ as the operator norm in `eq:odd-forcing-Gram`. The notebook bounds that operator by its Hilbert–Schmidt norm:

$$
\Theta_F^U\le\sqrt{\operatorname{tr}(GQ\,GF)}
=\texttt{sqrt(sum(GQ .* GF))}.
$$

The common whole-field sign between the loaded trials and manuscript inverse fields does not change GF. GQ already includes both retained-eigenvalue denominators; GF adds none.

A full-period W10 prior is evaluated on $\beta\in[0,2\pi]$. `F_β_prior` supplies explicit β bounds to `Linf_to_paras(...; ind=1)` on the padded physical rectangle. After β refinement, the θ seeds and mixed-derivative corrections use the refined β bounds of orders zero and two. Trim the ten layers only after refinement, then integrate GF over the whole physical space.

The code prints `Theta_F_U` and asserts its upper endpoint is at most `0.0638`, matching the paper bound in `eq:45-input-FU`.


In [8]:
# Global derivative priors for the complete weighted mixed forcing.
W_F_prior = let
    L_prior = dt(2) * Pi
    β_pt = grid(L_prior, 16000)
    θ_pt = grid(L1, 2000)
    F_lst = vec(mul_A_gradB(UP_phi_lst, up, β_pt, θ_pt))
    add_equal!.(F_lst, vec(mul_A_gradB(up, UP_phi_lst, β_pt, θ_pt)))

    W_F_prior = []
    for j in 1:K
        println("Forcing prior $j / $K")
        push!(W_F_prior, norm_lst(F_lst[j], "W$(k)Inf";
            paras=freq_forcing, domain=[L_prior,L1]))
    end
    W_F_prior
end
GC.gc(true)

Forcing prior 1 / 18
Forcing prior 2 / 18
Forcing prior 3 / 18
Forcing prior 4 / 18
Forcing prior 5 / 18
Forcing prior 6 / 18
Forcing prior 7 / 18
Forcing prior 8 / 18
Forcing prior 9 / 18
Forcing prior 10 / 18
Forcing prior 11 / 18
Forcing prior 12 / 18
Forcing prior 13 / 18
Forcing prior 14 / 18
Forcing prior 15 / 18
Forcing prior 16 / 18
Forcing prior 17 / 18
Forcing prior 18 / 18


In [9]:
# Whole-space forcing Gram, refined before removing the stencil layers.
GF = let
    β_pt = grid(L0, 12000, padding)
    θ_pt = grid(L1, 6000, padding)
    F_lst = vec(mul_A_gradB(UP_phi_lst, up, β_pt, θ_pt))
    add_equal!.(F_lst, vec(mul_A_gradB(up, UP_phi_lst, β_pt, θ_pt)))

    F_norms = []
    for j in 1:K
        println("Forcing refinement $j / $K")
        # β-only priors (component × 1 matrices) for the mixed-parity path
        F_β_prior = [reshape(W_F_prior[j][m, 1], :, 1) for m in 1: k+1]
        paras = Linf_to_paras(F_β_prior, k, [zero(dtype), freq_forcing[2]]; ind=1)
        push!(F_norms, norm_lst(F_lst[j], "W$(k-2)Inf";
            paras=paras, domain=[L0,L1], padding=padding))
        trim_border!(F_lst[j], padding)
    end

    GF = zeros(dtype, K, K)
    for i in 1:K, j in i:K
        println("Forcing Gram i = $i / $K, j = $j / $K")
        GF[i,j] = inner_product(F_lst[i], F_lst[j];
            W_norms_lst=[F_norms[i],F_norms[j]], domain=[L0,L1])
        GF[j,i] = GF[i,j]
    end
    Theta_F_U = sqrt(sum(GQ .* GF))
    println("Theta_F_U: ", Theta_F_U)
    @assert sup(Theta_F_U) <= inf(dt("0.0638"))
    GF
end
GC.gc(true)

Forcing refinement 1 / 18
Forcing refinement 2 / 18
Forcing refinement 3 / 18
Forcing refinement 4 / 18
Forcing refinement 5 / 18
Forcing refinement 6 / 18
Forcing refinement 7 / 18
Forcing refinement 8 / 18
Forcing refinement 9 / 18
Forcing refinement 10 / 18
Forcing refinement 11 / 18
Forcing refinement 12 / 18
Forcing refinement 13 / 18
Forcing refinement 14 / 18
Forcing refinement 15 / 18
Forcing refinement 16 / 18
Forcing refinement 17 / 18
Forcing refinement 18 / 18
Forcing Gram i = 1 / 18, j = 1 / 18
Forcing Gram i = 1 / 18, j = 2 / 18
Forcing Gram i = 1 / 18, j = 3 / 18
Forcing Gram i = 1 / 18, j = 4 / 18
Forcing Gram i = 1 / 18, j = 5 / 18
Forcing Gram i = 1 / 18, j = 6 / 18
Forcing Gram i = 1 / 18, j = 7 / 18
Forcing Gram i = 1 / 18, j = 8 / 18
Forcing Gram i = 1 / 18, j = 9 / 18
Forcing Gram i = 1 / 18, j = 10 / 18
Forcing Gram i = 1 / 18, j = 11 / 18
Forcing Gram i = 1 / 18, j = 12 / 18
Forcing Gram i = 1 / 18, j = 13 / 18
Forcing Gram i = 1 / 18, j = 14 / 18
Forcing Gram i

## Bound every pairwise convection term

List inputs to `mul_A_gradB` produce all $K\times K$ weighted fields $w(\phi_i\cdot\nabla)\phi_j$. Each ordered pair is retained: swapping i and j need not give the same field. The order-ten frequency table supplies the derivatives used by the physical L² quadrature, which needs orders through eight.

`inpd_tensor[i,j]` stores the square of the returned L² interval; `N[i,j]` takes its square root. The displayed aggregate is

$$
\left(\sum_{i,j=1}^{K}\|(\phi_i\cdot\nabla)\phi_j\|_{L^2(\mathbb R^3)}^2\right)^{1/2}.
$$

This calculation uses an independently created unpadded physical quarter grid. Its radial and angular reflection properties were checked above; it does not reuse the mixed-pressure `lfs` symmetry argument. A local `let` releases the pairwise sample fields after constructing the scalar and matrix results.
Once the tensor is complete, the cell prints `phi_product=sqrt(sum(inpd_tensor))` and checks it against `dt("0.679897")` before constructing N.


In [10]:
# Bound all ordered physical convection pairs, using their already weighted samples for L² integration.
inpd_tensor, N, phi_product = let
    # Independent whole-space grid, after earlier residual computations.
    N0, N1 = 6000, 3000
    β_pt = grid(L0, N0)
    θ_pt = grid(L1, N1)

    UP_i_nabla_UP_j_lst = mul_A_gradB(UP_phi_lst, UP_phi_lst, β_pt, θ_pt)

    inpd_tensor = zeros(dtype, K, K)
    for i in 1:K, j in 1:K
        UPvec = UP_i_nabla_UP_j_lst[i,j]
        UPvec_W_norms = norm_lst(UPvec, "W$(k)Inf"; paras=freq_pair, domain=[L0,L1])
        inpd_tensor[i, j] = norm_lst(UPvec, "L2"; paras=UPvec_W_norms, domain=[L0,L1])^2
    end

    phi_product = sqrt(sum(inpd_tensor))
    println("phi convection L2 aggregate: ", phi_product)
    @assert sup(phi_product) <= inf(dt("0.679897"))

    N = zeros(dtype, K, K)
    for i in 1:K, j in 1:K
        N[i,j] = sqrt(inpd_tensor[i, j])
    end

    inpd_tensor, N, phi_product
end
GC.gc(true)


phi convection L2 aggregate: [0.637858, 0.679897]_com


## Bounds checked in their calculation cells

The complete residual Gram gives `K7_U=sqrt(sum(GQ .* GR))`, preserving signed off-diagonal entries. `C_U` is checked by the symmetric spectral certificate; `A0_U` and `A1_U` use Gram spectral certificates. Gradient, convection, and mixed-forcing aggregates are printed and checked where computed. The final code cell saves the completed matrices and scalar aggregates.


### Save the completed Step 5 results

`step5_even_matrices.mat` contains `GQ`, `GR`, `G_phi`, `G_Dphi`, `GF`, `inpd_tensor`, `N`, `phi_grad`, `phi_product`, and `K7_U`.

`C_U`, `A0_U`, and `A1_U` are printed and checked inline. The triangle inequality gives $\|I+G^U\|\le1+A0_U$; that derived value is not saved. `Theta_F_U` is printed and checked against `0.0638`; GF is saved, while the trace-bound scalar is not.


In [11]:
# Save the completed matrices and bounds already checked in their calculation cells.
save_matrix("../data/generated/step5_even_matrices.mat", Dict(
    "GQ" => GQ,
    "GR" => GR,
    "G_phi" => G_phi,
    "G_Dphi" => G_Dphi,
    "GF" => GF,
    "inpd_tensor" => inpd_tensor,
    "N" => N,
    "phi_grad" => phi_grad,
    "phi_product" => phi_product,
    "K7_U" => K7_U,
))


## Relation to the paper's numerical inputs

| Computed bound | Notebook threshold | Rounded paper input |
|---|---|---|
| `K7_U` | `0.000453683` | $\Theta_r^U\le0.000454$ |
| `A0_U` | `4.28080` | $\Theta_\phi^U\le4.29$ |
| `A1_U` | `4.79779` | $\Theta_{\nabla\phi}^U\le4.8$ |
| `C_U` | `5.24693` | $\Theta_Q^U\le5.25$ |
| `phi_grad` | `0.369356` | Gradient aggregate ≤ $0.37$ |
| `phi_product` | `0.679897` | Convection aggregate ≤ $0.68$ |
| `Theta_F_U` | `0.0638` | $\Theta_F^U\le0.0638$ |

The paper combines these inputs in `eq:finite-rank-constants`, `eq:U-M3-definition`, and the profile and eigenpair energy estimates of Section `Sec:num prev`.
